# torch_rec

> A PyTorch copy of the PP-OCRv6 medium recognizer, so it can be trained.


In [ ]:
#| default_exp torch_rec


MAX compiles the recognizer for inference and has no gradients. This module is the same network written with `torch.nn`. It reads the same `RecognizerConfig` and `REC_BLOCKS`, defines its layers in the same order as `recognizer`, and uses the same tensor shapes. That lets the checkpoint's learned tensors move between the two by position.

The exported checkpoint is the merged inference form: convolutions carry biases and the backbone has no batch-norm, so there are no branches to rebuild. Only the SVTR neck has batch-norm, so train and eval modes differ there.


In [ ]:
#| hide
from nbdev import show_doc


In [ ]:
#| export
#| notest
from __future__ import annotations

from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from torch import nn

from solid_lens.config import REC_BLOCKS, Block, RecognizerConfig
from solid_lens.weights import checkpoint_arrays


## Blocks


A convolution is `nn.Conv2d`. Stride 1 keeps the width of the map, and a stride pair downsamples height only.


In [ ]:
#| export
#| notest
def conv(
    in_channels: int,
    out_channels: int,
    kernel: int | tuple[int, int],
    *,
    stride: int | tuple[int, int] = 1,
    padding: int | tuple[int, int] = 0,
    groups: int = 1,
    bias: bool = True,
) -> nn.Conv2d:
    return nn.Conv2d(
        in_channels, out_channels, kernel,
        stride=stride, padding=padding, groups=groups, bias=bias,
    )


The squeeze-excite gate is a hardsigmoid with slope `alpha`, written out because the exported graph uses 1/6 and a clamp.


In [ ]:
#| export
#| notest
class SqueezeExcite(nn.Module):
    "Global pool, 1x1 bottleneck, hardsigmoid gate."

    def __init__(self, channels: int, alpha: float, *, reduction: int = 4) -> None:
        super().__init__()
        self.alpha = alpha
        self.down = conv(channels, channels // reduction, 1)
        self.up = conv(channels // reduction, channels, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        pooled = x.mean(dim=(2, 3), keepdim=True)
        gate = (self.up(F.relu(self.down(pooled))) * self.alpha + 0.5).clamp(0, 1)
        return x * gate


In [ ]:
#| export
#| notest
def _unit_stride(stride: int | tuple[int, int]) -> bool:
    return stride == 1 if isinstance(stride, int) else tuple(stride) == (1, 1)


In [ ]:
#| export
#| notest
class LCNetV4Block(nn.Module):
    "Depthwise token mixer, optional SE, and a 2x pointwise channel mixer."

    def __init__(self, spec: Block, se_alpha: float) -> None:
        super().__init__()
        kernel, in_channels, out_channels, stride, use_se = spec
        self.residual = _unit_stride(stride) and in_channels == out_channels
        self.dw = conv(in_channels, in_channels, kernel, stride=stride,
                       padding=kernel // 2, groups=in_channels)
        self.se = SqueezeExcite(in_channels, se_alpha) if use_se else None
        self.expand = conv(in_channels, in_channels * 2, 1)
        self.compress = conv(in_channels * 2, out_channels, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        mixed = self.dw(x)
        if self.se is not None:
            mixed = self.se(mixed)
        channels = self.compress(F.gelu(self.expand(mixed)))
        return mixed + channels if self.residual else channels


In [ ]:
#| export
#| notest
def stage(blocks: list[Block], se_alpha: float) -> nn.Sequential:
    return nn.Sequential(*[LCNetV4Block(spec, se_alpha) for spec in blocks])


The stem has two branches that meet in a concat. The 2x2 convolutions and the max pool pad one row and column on the bottom and right, which is the checkpoint's SAME_UPPER.


In [ ]:
#| export
#| notest
class Stem(nn.Module):
    "Multi-branch stem. Total stride is 4."

    def __init__(self, mid: int, out: int) -> None:
        super().__init__()
        self.stem1 = conv(3, mid, 3, stride=2, padding=1)
        self.stem2a = conv(mid, mid // 2, 2)
        self.stem2b = conv(mid // 2, mid, 2)
        self.stem3 = conv(mid * 2, mid, 3, stride=2, padding=1)
        self.stem4 = conv(mid, out, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        pad = (0, 1, 0, 1)
        x = F.relu(self.stem1(x))
        branch = F.relu(self.stem2b(F.pad(F.relu(self.stem2a(F.pad(x, pad))), pad)))
        pooled = F.max_pool2d(F.pad(x, pad), 2, 1)
        merged = torch.cat([pooled, branch], dim=1)
        return F.relu(self.stem4(F.relu(self.stem3(merged))))


## LightSVTR


The neck mirrors `svtr`: a local depthwise mixer along the width, two global attention blocks at width 192, then an additive skip from the backbone features. Attention scales the queries by `1/sqrt(head_dim)`, which is what `scaled_dot_product_attention` does by default.


In [ ]:
#| export
#| notest
class ConvBNSwish(nn.Module):
    "1x1 projection used by the skip and the channel reduction."

    def __init__(self, in_channels: int, out_channels: int, eps: float) -> None:
        super().__init__()
        self.conv = conv(in_channels, out_channels, 1, bias=False)
        self.bn = nn.BatchNorm2d(out_channels, eps=eps)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return F.silu(self.bn(self.conv(x)))


In [ ]:
#| export
#| notest
class LocalMixer(nn.Module):
    "1xk depthwise conv along the text width, added back onto its input."

    def __init__(self, channels: int, kernel: int, eps: float) -> None:
        super().__init__()
        self.conv = conv(channels, channels, (1, kernel), padding=(0, kernel // 2),
                         groups=channels, bias=False)
        self.bn = nn.BatchNorm2d(channels, eps=eps)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + F.silu(self.bn(self.conv(x)))


In [ ]:
#| export
#| notest
class SVTRBlock(nn.Module):
    "Pre-norm global attention and a Swish MLP."

    def __init__(self, dim: int, num_heads: int, mlp_ratio: float, eps: float) -> None:
        super().__init__()
        if dim % num_heads:
            raise ValueError(f"{dim} channels is not divisible by {num_heads} heads")
        self.num_heads = num_heads
        self.norm1 = nn.LayerNorm(dim, eps=eps)
        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)
        self.norm2 = nn.LayerNorm(dim, eps=eps)
        hidden = int(dim * mlp_ratio)
        self.fc1 = nn.Linear(dim, hidden)
        self.fc2 = nn.Linear(hidden, dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self._attend(self.norm1(x))
        return x + self.fc2(F.silu(self.fc1(self.norm2(x))))

    def _attend(self, x: torch.Tensor) -> torch.Tensor:
        batch, width, channels = x.shape
        qkv = self.qkv(x).reshape(batch, width, 3, self.num_heads, channels // self.num_heads)
        query, key, value = qkv.permute(2, 0, 3, 1, 4).unbind(0)
        context = F.scaled_dot_product_attention(query, key, value)
        return self.proj(context.transpose(1, 2).reshape(batch, width, channels))


In [ ]:
#| export
#| notest
def _as_sequence(x: torch.Tensor) -> torch.Tensor:
    "NCHW with height 1 to (batch, width, channels)."
    return x.squeeze(2).permute(0, 2, 1)


In [ ]:
#| export
#| notest
class LightSVTR(nn.Module):
    "Local conv, global blocks, then an additive skip of the backbone features."

    def __init__(self, in_channels: int, dim: int, depth: int, num_heads: int,
                 mlp_ratio: float, local_kernel: int, eps: float, final_eps: float) -> None:
        super().__init__()
        self.skip = ConvBNSwish(in_channels, dim, eps)
        self.reduce = ConvBNSwish(in_channels, dim, eps)
        self.local = LocalMixer(dim, local_kernel, eps)
        self.blocks = nn.ModuleList(SVTRBlock(dim, num_heads, mlp_ratio, eps) for _ in range(depth))
        self.norm = nn.LayerNorm(dim, eps=final_eps)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        skip = _as_sequence(self.skip(x))
        sequence = _as_sequence(self.local(self.reduce(x)))
        for block in self.blocks:
            sequence = block(sequence)
        return self.norm(sequence) + skip


## Recognizer


`forward` returns CTC logits, shaped (batch, 40, 18710). The MAX graph ends in a softmax; `probabilities` is the same thing here. CTC loss wants log-probabilities, so training reads the logits.


In [ ]:
#| export
#| notest
class TorchRecognizer(nn.Module):
    "LCNetV4, a height pool, LightSVTR, and a CTC head. Layers are defined in the same order as `PPOCRV6Recognizer`."

    def __init__(self, config: RecognizerConfig | None = None) -> None:
        super().__init__()
        config = config or RecognizerConfig()
        self.config = config
        self.stem = Stem(config.stem_mid, config.stem_out)
        self.stage2 = stage(REC_BLOCKS["b2"], config.se_alpha)
        self.stage3 = stage(REC_BLOCKS["b3"], config.se_alpha)
        self.stage4 = stage(REC_BLOCKS["b4"], config.se_alpha)
        self.stage5 = stage(REC_BLOCKS["b5"], config.se_alpha)
        self.neck = LightSVTR(
            config.backbone_channels, config.svtr_dim, config.svtr_depth,
            config.svtr_heads, config.mlp_ratio, config.local_kernel,
            config.bn_eps, config.final_norm_eps,
        )
        self.head = nn.Linear(config.svtr_dim, config.num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.stage5(self.stage4(self.stage3(self.stage2(self.stem(x)))))
        # 3x2 average pool leaves one row of features per text column pair.
        return self.head(self.neck(F.avg_pool2d(x, (3, 2), (3, 2))))

    def probabilities(self, x: torch.Tensor) -> torch.Tensor:
        return self.forward(x).softmax(dim=-1)


In [ ]:
#| notest
from fastcore.test import test_eq

model = TorchRecognizer()
test_eq(sum(p.numel() for p in model.parameters()) > 10_000_000, True)
with torch.no_grad():
    out = model.eval()(torch.zeros(1, 3, 48, 320))
test_eq(tuple(out.shape), (1, 40, 18710))


## Weights


The learned tensors are the state dict without the batch-norm step counters. They are in the order `weights.arrays_from_graph` returns them, so a checkpoint loads by position. Shape and count are checked, so a mismatch fails here and not as a wrong answer later.


In [ ]:
#| export
#| notest
def _learned(model: nn.Module) -> list[tuple[str, torch.Tensor]]:
    return [(name, t) for name, t in model.state_dict().items() if not name.endswith("num_batches_tracked")]


In [ ]:
#| export
#| notest
def model_arrays(model: nn.Module) -> list[np.ndarray]:
    "Learned tensors as float32 arrays, in checkpoint order."
    return [t.detach().cpu().numpy().astype(np.float32, copy=True) for _, t in _learned(model)]


In [ ]:
#| export
#| notest
def set_arrays(model: nn.Module, arrays: list[np.ndarray]) -> None:
    "Copy checkpoint-ordered arrays into the model."
    slots = _learned(model)
    if len(arrays) != len(slots):
        raise ValueError(f"checkpoint has {len(arrays)} tensors, the model has {len(slots)}")
    with torch.no_grad():
        for (name, tensor), array in zip(slots, arrays, strict=True):
            if tuple(array.shape) != tuple(tensor.shape):
                raise ValueError(f"{name}: checkpoint shape {tuple(array.shape)} != model shape {tuple(tensor.shape)}")
            tensor.copy_(torch.from_numpy(np.ascontiguousarray(array)))


In [ ]:
#| export
#| notest
def load_recognizer(
    checkpoint: Path, # ONNX or `.npz` weights
    config: RecognizerConfig | None = None,
) -> TorchRecognizer:
    "A `TorchRecognizer` holding the weights in an ONNX or `.npz` checkpoint."
    model = TorchRecognizer(config)
    set_arrays(model, checkpoint_arrays(Path(checkpoint)))
    return model


In [ ]:
#| notest
other = TorchRecognizer()
set_arrays(other, model_arrays(model))
test_eq(all(torch.equal(a, b) for a, b in zip(model.state_dict().values(), other.state_dict().values())), True)


## Check against MAX


The two implementations have to agree on the official weights. This builds both from `medium_rec.onnx` and compares the softmax they produce on random input. It compiles the MAX graph, so it needs the checkpoint and is skipped unless run with the `notest` flag.


In [ ]:
#| notest
import numpy as np
from solid_lens.core import repo_root
from solid_lens.model import compile_recognizer

onnx_path = repo_root() / "models" / "medium_rec.onnx"
reference = load_recognizer(onnx_path).eval()
max_model = compile_recognizer(
    cache_dir=repo_root() / "models" / "native_medium_rec_48x320.mef")

def max_probs(x: np.ndarray) -> np.ndarray:
    return max_model.execute(np.ascontiguousarray(x))[0].to_numpy()

rng = np.random.default_rng(0)
x = rng.uniform(-1, 1, (1, 3, 48, 320)).astype(np.float32)
with torch.no_grad():
    ours = reference.probabilities(torch.from_numpy(x)).numpy()
theirs = max_probs(x)
print("max abs diff", float(np.abs(ours - theirs).max()))
test_eq(bool(np.allclose(ours, theirs, atol=1e-4)), True)
test_eq(ours.argmax(-1).tolist(), theirs.argmax(-1).tolist())
